# Pipeline v3

Iteration 3 of the lecture pipeline, built from scratch. This notebook documents every stage and script, shows how to
run each one, and records the decisions and runs made along the way.

Run it from the `pipeline/` folder with the repo's Python (no packages beyond the standard library are needed).

| Folder | What is in it |
|---|---|
| `input/<course>/input/` | `subs/` (SRT per lecture), `transcripts/`, `titles.tsv`, `course.json` (course title) |
| `input/<course>/media/` | lecture videos `L01.mp4` … |
| `prompts/` | prompt texts (`segmentation.txt`) |
| `configs/` | saved `LLMConfig` files |
| `scripts/` | `s0a_preprocess.py`, `s1a_segmentation.py`, `s1b_verify_segments.py`, `s1c_segment_texts.py`; helpers `llm/`, `view.py` |
| `output/<course>/` | one folder per step: `stage_0a_preprocess/`, `stage_1a_segmentation/`, `stage_1b_verify_segments/`, `stage_1c_segment_texts/` |

Stages so far:

| Stage | Script | Input | Output |
|---|---|---|---|
| 0a Preprocess | `s0a_preprocess.py` | SRT files | sentences with IDs, and their times |
| 1a Segmentation | `s1a_segmentation.py` | 0a sentences | the model's segments (start sentence + sub-goal), checked by 1b's checks with retries |
| 1b Verify segments | `s1b_verify_segments.py` | 1a answers | deterministic checks; segment maps (segment → sentence range, time, sub-goal) |
| 1c Segment texts | `s1c_segment_texts.py` | 1b segment maps | one `.txt` per segment |
| – Browsing | `view.py` | all outputs | dicts for the notebook (`sentences["1/0"]`, `segments["1/S0"]`) |

## Record: decisions so far

**Inputs (2026-11-09).** The VR course (NPTEL *Foundation for Virtual and Augmented Reality Systems*, 63 lectures) is
in `input/nptel_vr/`. A copy stays in `dataset/virtual_and_augmented_reality/`. Every course's subtitles and
transcripts (NPTEL and MIT) are human-written and verified, so their punctuation can be trusted.

- Lectures 4 and 5 have no NPTEL subtitle file. Their `lec_4.srt` / `lec_5.srt` come from pipeline v1: the official
  transcript, one sentence per cue, timed by aligning it to YouTube's automatic captions.
- Videos: `input/nptel_vr/media/L01.mp4` … `L63.mp4`.

**Earlier segmentations not reused.** v1's segments and v2's (large run, Qwen 3.8 27B) were looked at. v2's large run
had 321 parts, 36 of them over 10 min and one of 28 min (L51/P04: a whole worked problem that the "stand alone" rule
would not cut). Run 1 started again from v2's prompt unchanged.

**Segmentation prompt rewritten (2026-11-11, used from run 2).** A segment is defined by its sub-goal:
1. A segment is where the lecturer completes one sub-goal with respect to the whole lecture. Greetings and endings are
   not sub-goals (they are not lecture content): opening remarks go into the first segment, closing remarks into the last.
2. Cut where the sub-goal is achieved and closed, not at a bridge: the bridge goes to the next segment. Each segment must
   stand alone and be watchable independently: no deixis (no pointing back to the previous segment).
3. No length limit. (A 6–15 min rule was considered and dropped: run 1 had 473 of 545 segments under 6 min, so code
   would have done nearly all the segmenting.)

Added: course and lecture titles in the prompt; sub-goals at the level of the lecture's sections; a working method
(goal → sub-goals → cuts → check); the answer gives each segment's start and its sub-goal, so the model commits to what
each segment achieves and we can judge segments at a glance.

**Deterministic checks only (stage 1b).** Code checks the answer and the segments (as sets of sentences: exhaustive,
mutually exclusive, contiguous and ordered, non-empty). It does not merge, split or re-segment. A failed check is sent
back to the model with the error and the model answers again.

**Sentence splitting.** Our own splitter (stage 0) was compared with pySBD on the VR course and three MIT courses.
pySBD was worse everywhere:

| Course | Splitter | Sentences | Over 100 words | Longest |
|---|---|--:|--:|--:|
| VR | ours / pySBD | 11,893 / 12,139 | 3 / 3 | 124 / 124 |
| MIT 18.06 | ours / pySBD | 15,945 / 9,678 | 0 / 284 | 96 / 4,008 |
| MIT 6.7960 | ours / pySBD | 18,326 / 14,027 | 1 / 248 | 120 / 4,525 |
| MIT 9.00SC | ours / pySBD | 19,739 / 15,950 | 1 / 153 | 103 / 2,358 |

On VR, pySBD split at 281 stray mid-sentence full stops (leaving fragments that start in lowercase) and 47 numbers
read as list markers, and missed 87 real ends after acronyms and variables ("AR, VR, and MR. So…"). On MIT text it
stops splitting after `--` dashes. Known gaps of our splitter, for later: MIT sentences that start with a lowercase
variable ("e is this vector"), speaker labels (`PROFESSOR:`), file names like `lec_24b.srt`, the OCW licence notice.

**Segmentation is in a testing phase.** The model's segments are used as they are (no merging, no windows for long
lectures). What to change is decided after looking at the outputs.

**Names (2026-11-11).** Steps are numbered by stage and step: 0a preprocess, 1a segmentation, 1b verify segments,
1c segment texts. Python file names cannot start with a digit, so scripts are `s0a_…`, `s1a_…`; output folders
`stage_0a_…`, `stage_1a_…`.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path
from IPython.display import Markdown, display

import sys
assert Path("scripts").is_dir(), "run this notebook from the pipeline folder"
sys.path.insert(0, str(Path.cwd()))   # so that "from scripts import ..." works
COURSE = "nptel_vr"
OUT = Path("output") / COURSE

## Stage 0a: preprocess

`scripts/s0a_preprocess.py` turns each lecture's SRT into numbered sentences.

- All cue text of a lecture is joined and split at sentence ends (`.` `?` `!`, plus any closing quote or bracket) that
  are followed by a space and a capital letter, digit or opening quote. A sentence running across cues stays whole; a
  cue holding several sentences is split. `Dr.`, `Prof.`, `Mr.`, `e.g.`, `i.e.`, `vs.`, `Fig.`, `Eq.` do not end a
  sentence. `MR.` does (in this course it is Mixed Reality).
- Sentence IDs are `<lecture>/<i>`, with `i` from 0 within the lecture: `1/0`, `1/1`, …

| File | Content |
|---|---|
| `output/<course>/stage_0a_preprocess/lec_<n>.json` | `{"1/0": "Hello, welcome to …", …}` |
| `output/<course>/stage_0a_preprocess/times/lec_<n>.json` | `{"1/0": {"start": 24.297, "end": 31.884}, …}` |

**Times.** A sentence that begins at the start of a cue gets that cue's start time; one that ends at the end of a cue
gets that cue's end time. When a sentence boundary falls inside a cue, that side has no time of its own and is `null`.
A `null` start means the sentence begins inside the cue where the previous sentence ends (whose end is `null` too). In
the VR course 201 sentences start inside a cue (38 of them also end inside one). No times are made up.

Command line: `python scripts/s0a_preprocess.py nptel_vr`

In [ ]:
from scripts import s0a_preprocess
stats = s0a_preprocess.run(COURSE)

In [ ]:
s0 = OUT / "stage_0a_preprocess"
sents = json.loads((s0 / "lec_1.json").read_text())
times = json.loads((s0 / "times" / "lec_1.json").read_text())
for k in list(sents)[:3]:
    print(k, times[k], sents[k])
print("\nboundaries inside a cue (null times) in lecture 1:")
for k, t in times.items():
    if None in t.values():
        print(k, t, sents[k][:80])

## The LLM API (`scripts/llm/`)

One API for **vLLM** and **llama.cpp**. Both serve the OpenAI chat API, so requests are sent the same way. The parts:

| Name | What it does |
|---|---|
| `LLMConfig` | every setting: engine, model, where the weights are, server flags, sampling, thinking, client |
| `LLM` | starts or attaches to the server, sends requests (`generate`, `generate_many`), caches and logs |
| `Request` | one prompt and its options (schema, check, overrides) |
| `Result` | the answer, thinking, parsed JSON, tokens, time, errors |
| `Server` | builds the server command, starts and stops it (used by `LLM`) |

**Launch or attach.** `LLM(cfg).start()` attaches to a server that already answers at the URL; otherwise (with
`launch=True`) it starts one, waits until it is ready, and `stop()` shuts it down again. A server it attached to is
left running. Set `url` (and `launch=False`) to use a server on another machine, e.g. the EC2.

**Sequential and parallel.** `generate_many(requests, parallel=False)` sends one request at a time.
`parallel=True` sends them all at once, up to `concurrency` (default `slots`) in flight, and the engine batches them:
vLLM by continuous batching, llama.cpp over its slots. Results come back in the order of the requests.

**On this machine** llama.cpp runs inside the `llm-engine` toolbox (`toolbox="llm-engine"`). vLLM runs on the EC2.

### `LLMConfig` fields

| Group | Field | Default | Meaning |
|---|---|---|---|
| model | `engine` | `"llamacpp"` | `"vllm"` or `"llamacpp"` |
| | `model` | `""` | name requests use (vLLM `--served-model-name`, llama.cpp `--alias`); empty: `model_path` (vLLM) or what the server reports |
| | `model_path` | `""` | weights. vLLM: HF repo id or local folder. llama.cpp: local `.gguf`, `"org/repo:file.gguf"`, or `"org/repo:QUANT"` |
| server | `launch` | `True` | start a server when none answers; `False`: attach only |
| | `host`, `port` | `127.0.0.1`, `8000` | where the server listens |
| | `url` | `""` | a server elsewhere, e.g. `"http://10.0.0.5:8000"` (overrides host and port; never launched) |
| | `binary` | `""` | `vllm` / `llama-server` executable; empty: found on PATH |
| | `toolbox` | `""` | run the server inside this toolbox container (here: `"llm-engine"`) |
| | `env` | `{}` | extra environment variables for the server |
| | `startup_timeout_s` | `1800` | how long to wait for the server to be ready |
| | `server_log` | `""` | server output file; empty: `llm_server_<port>.log` |
| engine | `context_length` | `32768` | vLLM `--max-model-len`; llama.cpp `-c` (one pool shared by the slots with `kv_unified`) |
| | `slots` | `8` | requests run at once: vLLM `--max-num-seqs`, llama.cpp `-np` |
| | `seed` | `0` | server seed; a request's seed is `seed + attempt` |
| | `extra_args` | `[]` | more server flags, passed as they are |
| vLLM | `gpu_memory_utilization` | `0.90` | share of GPU memory vLLM takes |
| | `tensor_parallel_size` | `1` | GPUs per model |
| | `dtype`, `quantization`, `kv_cache_dtype` | `""` | e.g. `"bfloat16"`, `"fp8"`, `"fp8"`; empty: vLLM's choice |
| | `prefix_caching` | `True` | reuse the KV cache of a shared prompt start |
| | `max_num_batched_tokens` | `0` | 0: vLLM's default |
| | `reasoning_parser` | `""` | e.g. `"qwen3"`, `"gemma4"`: thinking returned apart from the answer |
| vLLM + llama.cpp | `think_end` | `""` | words that close the thinking when it reaches `think_budget` |
| llama.cpp | `gpu_layers` | `99` | `-ngl` |
| | `flash_attn` | `True` | `-fa on` |
| | `kv_unified` | `True` | one KV pool for all slots |
| | `jinja` | `True` | the model's own chat template |
| | `reasoning_format` | `"deepseek"` | thinking returned in `reasoning_content` |
| sampling | `temperature`, `top_p`, `top_k`, `min_p` | `0.6`, `0.95`, `20`, `0.0` | per request; a `Request` can override |
| | `presence_penalty`, `repetition_penalty` | `0.0`, `1.0` | |
| | `max_tokens` | `8192` | thinking + answer; 0: no cap |
| thinking | `thinking` | `True` | `enable_thinking` in the chat template (llama.cpp: `--reasoning on/off`) |
| | `think_budget` | `0` | cap on thinking tokens, 0: none. vLLM: per request; llama.cpp: per server (fixed at start) |
| client | `concurrency` | `0` | parallel mode: requests in flight; 0: `slots` |
| | `timeout_s` | `3600` | one request |
| | `retries` | `2` | more attempts when an answer fails its check or is not valid JSON (next seed each time) |
| | `cache_file` | `""` | answers cached by request hash (JSON lines); the same request is answered from the cache |
| | `log_file` | `""` | one JSON line per attempt: tokens, time, finish, errors |

Methods: `cfg.with_(**changes)` (a changed copy), `cfg.save(path)` (JSON), `LLMConfig.load(path)` (`.json` or
`.toml`), `cfg.to_dict()`, `cfg.base_url`.

In [ ]:
from scripts.llm import LLM, LLMConfig, Request, Result, Server

# The config used for run 1: Gemma 4 E4B (Q8 GGUF, already downloaded) on llama.cpp in the llm-engine toolbox.
local = LLMConfig.load("configs/gemma4_e4b_llamacpp.json")
print(json.dumps({k: v for k, v in local.to_dict().items() if v != LLMConfig().to_dict()[k]}, indent=1))
print("\nserver command:\n", " ".join(Server(local).command()))

In [ ]:
# vLLM on the EC2 (v2's large setup: Qwen 3.8 27B FP8). Only builds the command here; nothing is started.
ec2 = LLMConfig(
    engine="vllm",
    model_path="Qwen/Qwen3.8-27B-FP8",
    binary="~/vllm-venv/bin/vllm",
    context_length=65536, slots=128, gpu_memory_utilization=0.92,
    reasoning_parser="qwen3",
    think_end="\n\nI have thought enough; now I give the final answer.\n</think>\n\n",
    think_budget=8192, max_tokens=0,
    temperature=0.6, top_p=0.95, top_k=20,
    extra_args=["--language-model-only"],
)
print(" ".join(Server(ec2).command()))

# Attach to a server that is already running somewhere (no launch):
remote = LLMConfig(engine="vllm", url="http://<ec2-address>:8000", launch=False)

# A changed copy, and saving / loading:
greedy = local.with_(temperature=0.0, think_budget=4096)
# greedy.save("configs/my_config.json"); LLMConfig.load("configs/my_config.json")

### Sending requests

`Request` fields: `prompt`, `key` (your ID, shown in results and logs), `system`, `schema` (JSON schema: the answer is
constrained to it and parsed into `result.parsed`), `check` (a function of the parsed answer that returns a list of
problems; any problem means another attempt), `max_tokens`, `think_budget`, `sampling` (overrides such as
`{"temperature": 0}`), `meta` (returned on the result).

`Result` fields: `key`, `content` (the answer), `reasoning` (the thinking), `parsed`, `ok`, `errors`, `finish`
(`"length"` = cut off by `max_tokens`), `input_tokens`, `output_tokens`, `latency_s`, `attempts`, `cached`, `hash`,
`meta`.

`generate` and `generate_many` take `cache_file=` and `log_file=` to override the config's (stages use their own, e.g.
`output/<course>/stage_1a_segmentation/llm/`). `llm.totals` counts requests, attempts, cache hits and tokens.

The cells below need a running model server (they start the local one, about 9 GB of GPU memory).

In [ ]:
llm = LLM(local).start()          # attaches if it is already running
print(llm.model, "| tokens in 'Hello world':", llm.count_tokens("Hello world"))

r = llm.generate("List the prime numbers between 80 and 120.", max_tokens=4000)
print(r.content, "|", r.output_tokens, "tokens,", f"{r.latency_s:.1f} s, thinking: {len(r.reasoning)} chars")

In [ ]:
# A JSON answer, parsed and checked. A failed check is retried with the next seed (up to cfg.retries times).
primes = Request(
    prompt="List the prime numbers between 80 and 120 as a JSON list of integers.",
    key="primes",
    schema={"type": "array", "items": {"type": "integer"}},
    check=lambda xs: [] if all(80 < x < 120 for x in xs) else ["a number is out of range"],
)
r = llm.generate(primes)
print(r.ok, r.parsed, r.attempts, r.errors)

In [ ]:
# Many requests: one at a time, then all at once.
import time
reqs = [Request(prompt=f"What is {a} times {b}? Answer with the number only.", key=f"{a}x{b}", max_tokens=2000)
        for a, b in [(12, 13), (17, 19), (23, 29), (31, 37), (41, 43), (47, 53)]]
t = time.time(); seq = llm.generate_many(reqs, parallel=False); print(f"sequential: {time.time() - t:.1f} s")
t = time.time(); par = llm.generate_many(reqs, parallel=True); print(f"parallel:   {time.time() - t:.1f} s")
print([(x.key, x.content) for x in par])

In [ ]:
llm.stop()   # stops the server only if this LLM started it

## Stage 1a: segmentation

`scripts/s1a_segmentation.py` splits each lecture into segments with the prompt in `prompts/segmentation.txt`. The
prompt is filled with the course title (`input/<course>/input/course.json`), the lecture title (`titles.tsv`) and the
whole lecture, numbered from `[1]`. The model answers with its segments in order (constrained by a JSON schema):

```json
[{"start": 1, "subgoal": "Explain what extended reality means and how VR, AR and MR relate to it."},
 {"start": 38, "subgoal": "Describe how a head-mounted display creates a sense of presence."}]
```

Every answer is checked with stage 1b's checks (`s1b_verify_segments.check`). When a check fails, the model is shown
its answer and the problems ("The first segment must start at sentence 1, not at sentence 8.") and answers again, up to
`retries` times (`Request(feedback=True)` in the LLM API).

**Parallel** means every lecture is sent at once and the engine batches them; `parallel=False` sends one at a time.

| File | Content |
|---|---|
| `stage_1a_segmentation/raw/lec_<n>.json` | `answer`, `ok`, `errors`, `attempts`, `history` (earlier answers and their problems), tokens, time, `reasoning` |
| `stage_1a_segmentation/llm/cache.jsonl`, `calls.jsonl` | answer cache (a re-run is answered from it) and one line per call |

`run(course, llm, lectures=None, parallel=True, max_tokens=None, think_budget=None)`; the last two override the
config for this stage.

Command line: `python scripts/s1a_segmentation.py nptel_vr --llm configs/gemma4_e4b_llamacpp.json [--only 1-5] [--sequential]`
(starts the server, or attaches to a running one, and stops what it started).

In [ ]:
from scripts import s1a_segmentation
print(s1a_segmentation.PROMPT.read_text())

In [ ]:
with LLM(local) as llm:
    seg_summary = s1a_segmentation.run(COURSE, llm)              # every lecture, all at once
    # s1a_segmentation.run(COURSE, llm, lectures=[1, 2], parallel=False)
seg_summary

## Stage 1b: verify segments

`scripts/s1b_verify_segments.py`: deterministic checks only, in code. It never merges, splits or re-segments.

Checks on the answer: a non-empty list; each item has an integer `start` and a non-empty `subgoal`; the first segment
starts at sentence 1; every start is within the lecture; starts strictly increase.

Checks on the segments built from it, each segment a set of sentences:

| # | Check | Meaning |
|---|---|---|
| 1 | exhaustive | every sentence belongs to some segment |
| 2 | mutually exclusive | no sentence belongs to two segments |
| 3 | contiguous and ordered | each segment is a run of consecutive sentences, following the previous one |
| 4 | non-empty | every segment has at least one sentence |

An answer that passes the answer checks always passes 1–4 (segments are built from ordered starts); 1–4 guard against
bugs. Stage 1a runs `check()` on every answer and retries with the errors; this stage checks the saved answers again and
writes the segment maps.

| File | Content |
|---|---|
| `stage_1b_verify_segments/segment_maps/lec_<n>.json` | `{"1/S0": {"first": "1/0", "last": "1/41", "start": 24.297, "end": 292.435, "duration_s": 268.138, "subgoal": "Explain …"}, …}` |
| `stage_1b_verify_segments/report.json` | per lecture: `ok`, `errors`, `attempts`, `fixed_on_retry` (the problems a retry fixed) |

Segment IDs are `<lecture>/S<k>`, `k` from 0. `start` is the first sentence's start and `end` the last sentence's end;
either is `null` when that boundary falls inside a subtitle cue (see stage 0a), and `duration_s` is then `null` too.

Command line: `python scripts/s1b_verify_segments.py nptel_vr [--only 1-5]`

In [ ]:
from scripts import s1b_verify_segments
verify_summary = s1b_verify_segments.run(COURSE)
ids = [f"1/{i}" for i in range(10)]
print(s1b_verify_segments.check([{"start": 2, "subgoal": "x"}, {"start": 2, "subgoal": ""}], ids))

## Stage 1c: segment texts

`scripts/s1c_segment_texts.py` writes one text file per segment from the segment maps:
`stage_1c_segment_texts/lec_<n>/S<k>.txt`, one line per sentence, `[<lecture>/<sentence>] text`.
`write(course, lectures=None)`; command line: `python scripts/s1c_segment_texts.py nptel_vr [--only 1-5]`.

In [ ]:
from scripts import s1c_segment_texts
s1c_segment_texts.write(COURSE)
print((OUT / "stage_1c_segment_texts/lec_1/S0.txt").read_text()[:600])

## Browsing the outputs (`scripts/view.py`)

`Course(course, lectures=None)` loads every stage that has output into dicts, for looking at in this notebook:

| | Gives |
|---|---|
| `c.sentences["1/0"]` | the sentence text |
| `c.times["1/0"]` | `{"start": …, "end": …}` |
| `c.segments["1/S0"]` | the segment, shown as Markdown (header with sentences and time, its sub-goal, then each sentence); `.text` is the plain text, `.sentences` a dict, `.subgoal` |
| `c.segments_of(1)` | the segments of lecture 1 |
| `c.lecture(1)` | a Markdown table of lecture 1's segments with their sub-goals |
| `c.lectures` | lecture numbers |

Times show as m:ss; `?` is a boundary inside a subtitle cue.

In [ ]:
from scripts.view import Course
c = Course(COURSE)
c

In [ ]:
c.sentences["1/0"], c.times["1/0"]

In [ ]:
c.lecture(1)

In [ ]:
c.segments["1/S0"]

## Runs

### Run 1 (2026-11-09): Gemma 4 E4B, local, full course, v2's prompt

Config: `configs/gemma4_e4b_llamacpp.json`: Gemma 4 E4B (unsloth `UD-Q8_K_XL` GGUF) on llama.cpp in the
`llm-engine` toolbox, RTX 5070 12 GB; 6 slots sharing a 131,072-token context; Gemma's own sampling (temperature 1.0,
top_p 0.95, top_k 64); thinking on, capped at 8,192 tokens; `max_tokens` 12,288 (both as in v2). Every lecture sent
at once. Prompts: 5,116 tokens on average, 8,246 at most (lecture 39).

**Results.** 63/63 lectures segmented in 355 s (5.9 min), no failed answers, no retries, nothing cut off by
`max_tokens`, no ignored cut numbers. Thinking: 1,500 output tokens per lecture on average (2,487 at most), well under
the 8,192 cap.

| | Run 1 (Gemma 4 E4B, raw cuts) | v2 large (Qwen 3.8 27B, after merging parts under 2 min) |
|---|--:|--:|
| Segments | 557 (4–16 per lecture, 8.8 on average) | 321 |
| Median duration | 2.3 min | 4.6 min |
| Under 1 min / under 2 min | 77 / 229 | 0 / 0 (merged) |
| Over 10 min / over 15 min | 20 / 2 | 36 / 10 |
| Longest | 18.3 min (38/S2) | 27.9 min (L51/P04) |

Observations:
- Many very short segments are the lecture's closing lines cut off on their own ("So, looking forward to seeing you in
  the next part of the lecture.": 10/S3, 7.6 s), or a short aside.
- 6 cuts fall on a sentence that starts inside a subtitle cue, so 12 segments have a `null` start or end: 1/S4–S5,
  13/S1–S2, 42/S11–S12, 48/S10–S11, 57/S5–S6, 59/S5–S6.


### Run 2 (2026-11-11): Gemma 4 E4B, local, full course, new prompt

Same model and config as run 1; the new segmentation prompt (sub-goals, bridges, stand-alone, opening and closing
remarks); answers checked by stage 1b with retries.

**Results.** 63/63 lectures pass 1b, 622 segments, 594 s (9.9 min). 6 lectures were fixed on retry, all with the same
error: the model left the opening remarks out ("The first segment must start at sentence 1, not at sentence 8"; lectures
14, 18, 22, 45, 48, 54); one retry fixed each. Thinking: 2,378 output tokens per lecture on average (4,261 at most).

| | Run 1 (v2 prompt) | Run 2 (new prompt) |
|---|--:|--:|
| Segments | 557 | 622 |
| Per lecture | 4–16 (mean 8.8) | 5–25 (mean 9.9) |
| Median duration | 2.3 min | 2.3 min |
| Under 1 min / under 2 min | 77 / 229 | 62 / 250 |
| Over 10 min / over 15 min | 20 / 2 | 6 / 1 |
| Longest | 18.3 min | 15.4 min |
| First segment under 2 min | 41 | 29 |
| Last segment under 2 min | 48 | 36 |
| Segments with a `null` time | 12 | 10 |
| Open with a back-reference* | 59 | 55 |
| End on a bridge* | 39 | 34 |

\* rough word-pattern counts ("This…", "So, that…" at a segment start; "Now, let us…", a question at a segment end),
not checks. About two thirds of the cuts agree between the runs (run 2 cuts within 3 sentences of a run 1 cut: 363 of
559).

What the sub-goals show:
- The model treats the opening and closing remarks as sub-goals and labels them as such: a 1-min first segment of
  greeting and recap becomes "Summarize the different types of elements that can be tracked" (17/S0); a 1-min closing
  becomes "Conclude the lecture and identify the topics of subsequent lessons" (5/S6). The rule "never a segment of
  their own" is followed in about half the lectures.
- Items of one list are still cut apart: the stages of a contextual inquiry become 41 s and 11 s segments (36/S3, 36/S4).
- Long worked problems are now split by sub-goal: lecture 51 (v2's 28-min segment) has 10 segments, the longest 7.5 min
  ("Formulate the minimum time path problem using a graph", "Define the edge weights", "Calculate the travel time…",
  "Adapt … into a variation of the TSP", "Detail the algorithmic steps…").

In [ ]:
# Segment statistics for the current outputs (run 2).
import statistics as st
raws = [json.loads(p.read_text()) for p in sorted((OUT / "stage_1a_segmentation/raw").glob("lec_*.json"),
                                                  key=lambda p: int(p.stem.split("_")[1]))]
segs = list(c.segments.values())
durs = [s.duration_s / 60 for s in segs if s.duration_s is not None]
print(f"lectures ok {sum(r['ok'] for r in raws)}/{len(raws)}, segments {len(segs)}, "
      f"per lecture {len(segs) / max(1, len(c.lectures)):.1f}")
print(f"duration (min): median {st.median(durs):.1f}, min {min(durs):.1f}, max {max(durs):.1f}; "
      f"over 10 min: {sum(d > 10 for d in durs)}, over 15: {sum(d > 15 for d in durs)}, under 1: {sum(d < 1 for d in durs)}")
print("segments with a time inside a cue (null):", [s.id for s in segs if s.duration_s is None])
print("fixed on retry:", {r["lecture"]: r["history"][0]["errors"] for r in raws if r.get("history")})
print("cut off at max_tokens:", [r["lecture"] for r in raws if r["finish"] == "length"])
print("output tokens: mean", round(st.mean(r["output_tokens"] for r in raws)), "max", max(r["output_tokens"] for r in raws))
for s in sorted(segs, key=lambda s: -(s.duration_s or 0))[:5]:
    print(f"  longest: {s.id} {s.duration_s / 60:.1f} min, {len(s.sentences)} sentences")